# 08. Эластичность: динамика и сегментный разбор

Углублённый разбор поверх калиброванной модели (`05`). Уникально относительно 06/07:
1. **динамика area-взвешенной эластичности во времени** (поквартально) + ключевая ставка;
2. сегментный разрез с **фильтром надёжности по `q_base`** (малая база → мусорная ε);
3. сравнение с прозрачным **логит-бейзлайном**.

Эластичность везде — area-взвешенная `E=ΔQ/(Q·Δ)`, `Q=Σ p·area`, δ=±5%.

In [ ]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd if (_cwd / "data").is_dir() else _cwd.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.config import MODEL_DATASET_PARQUET, CALIBRATED_MODEL_PATH, PROCESSED_DIR, TARGET_COL, PRICE_COL
from src.modeling import CalibratedModel  # noqa: F401
from src.elasticity import apply_price_shock  # единый движок шока цены (см. src/elasticity.py)

C1, C2, C3 = "#2C7BB6", "#D7542B", "#2D9E5F"
plt.rcParams.update({"figure.dpi": 130, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.right": False, "axes.spines.top": False})
FIG_DIR = REPO_ROOT / "reports" / "figures" / "08_segments"
FIG_DIR.mkdir(parents=True, exist_ok=True)

HORIZON_CUT = pd.Timestamp("2025-04-30")
DELTA = 0.05

model: CalibratedModel = joblib.load(CALIBRATED_MODEL_PATH)
FEATURES = model.features

def predict_p(m, X):
    return m.predict_proba(X[FEATURES])[:, 1]

# Локальный apply_price_shock удалён — используем единый src.elasticity.apply_price_shock
# (двигает только признаки УРОВНЯ цены; price_change_* и коллинеарные относительные заморожены),
# чтобы 06/07/08 считали эластичность одинаково.

def demand_elasticity(data, delta=DELTA):
    """area-взвеш. ε и база Q (м²) для среза."""
    X, a = data[FEATURES], data["area"].to_numpy()
    Q0 = (predict_p(model, X) * a).sum()
    Qm = (predict_p(model, apply_price_shock(X, -delta)) * a).sum()
    Qp = (predict_p(model, apply_price_shock(X,  delta)) * a).sum()
    return (Qp - Qm) / (Q0 * 2 * delta), Q0

df = pd.read_parquet(MODEL_DATASET_PARQUET)
lab = df[(df["is_sold_past"] == 0) & (df["file_date"] <= HORIZON_CUT)].copy()
REF = lab["file_date"].max()
cur = lab[lab["file_date"] == REF].copy()
print(f"labelable активных строк: {len(lab):,} | опорный срез {REF.date()}: {len(cur):,} лотов")

## 1. Динамика эластичности во времени (поквартально) + ключевая ставка

In [ ]:
lab["q"] = lab["file_date"].dt.to_period("Q")
rows = []
for q, sub in lab.groupby("q", observed=True):
    ref = sub["file_date"].max()
    cs = sub[sub["file_date"] == ref]
    if len(cs) < 500:
        continue
    e, Q0 = demand_elasticity(cs)
    rows.append(dict(quarter=str(q), elast=e, key_rate=cs["key_rate"].mean(), n=len(cs)))
dyn = pd.DataFrame(rows)
display(dyn.round({"elast": 3, "key_rate": 2}))

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(dyn["quarter"], dyn["elast"], "o-", color=C1, lw=2, label="эластичность (area-взвеш.)")
ax.axhline(-1, color="0.6", ls=":", lw=1, label="ε=−1 (граница эластичности)")
ax.set_ylabel("ε", color=C1); ax.set_xlabel("квартал"); ax.tick_params(axis="x", rotation=45)
ax2 = ax.twinx()
ax2.plot(dyn["quarter"], dyn["key_rate"], "s--", color=C2, lw=1.5, label="ключевая ставка, %")
ax2.set_ylabel("ключевая ставка, %", color=C2); ax2.grid(False)
l1, lb1 = ax.get_legend_handles_labels(); l2, lb2 = ax2.get_legend_handles_labels()
ax.legend(l1 + l2, lb1 + lb2, loc="lower left", fontsize=8)
plt.tight_layout(); fig.savefig(FIG_DIR / "fig01_elasticity_dynamics.png", bbox_inches="tight"); plt.show()

print(f"corr(эластичность, ключевая ставка): {dyn['elast'].corr(dyn['key_rate']):.3f}")

## 2. Сегментный разрез с фильтром надёжности по q_base

In [ ]:
def segment_elasticity_table(data, group_cols, delta=DELTA, min_qbase=50.0):
    X, a = data[FEATURES], data["area"].to_numpy()
    p0 = predict_p(model, X); pm = predict_p(model, apply_price_shock(X, -delta)); pp = predict_p(model, apply_price_shock(X, delta))
    t = data[group_cols].copy()
    t["q0"], t["qm"], t["qp"] = p0 * a, pm * a, pp * a
    g = t.groupby(group_cols, observed=True).agg(q0=("q0","sum"), qm=("qm","sum"), qp=("qp","sum"),
                                                 n=("q0","size")).reset_index()
    g["elast"] = (g["qp"] - g["qm"]) / (g["q0"] * 2 * delta)
    g["reliable"] = g["q0"] >= min_qbase          # фильтр: малая ожидаемая база → мусорная ε
    return g.sort_values("elast")

seg = segment_elasticity_table(cur, ["region", "project_class"])
print("Регион × класс (area-взвеш. ε):")
display(seg[["region","project_class","n","q0","elast","reliable"]].round({"q0":0,"elast":3}))

room = segment_elasticity_table(cur, ["region", "room_count"])
room = room[room["region"].isin(["Москва","Новая Москва"])]

fig, ax = plt.subplots(figsize=(10, 5))
rel = seg[seg["reliable"]].sort_values("elast")
ax.barh(rel["region"] + " · " + rel["project_class"], rel["elast"], color=C1)
ax.axvline(-1, color=C2, ls="--", lw=1, label="ε=−1")
ax.set_xlabel("area-взвеш. ε"); ax.set_title("Эластичность по регион × класс (q_base≥50 м²)"); ax.legend()
plt.tight_layout(); fig.savefig(FIG_DIR / "fig02_elasticity_region_class.png", bbox_inches="tight"); plt.show()

n_unreliable = (~seg["reliable"]).sum()
if n_unreliable:
    print(f"⚠️ отфильтровано {n_unreliable} сегментов с q_base<50 м² (ε ненадёжна):")
    display(seg[~seg["reliable"]][["region","project_class","n","q0","elast"]].round({"q0":1,"elast":2}))

## 3. Сравнение с логит-бейзлайном

In [ ]:
# прозрачный бейзлайн: logit(target ~ log_price + сегментные дамми), та же area-взвеш. ε через шок цены
base = cur.dropna(subset=["log_price", TARGET_COL]).copy()
ctrl = pd.get_dummies(base[["region", "project_class", "room_count"]].astype(str), drop_first=True)
Xb = pd.concat([base[["log_price"]].reset_index(drop=True), ctrl.reset_index(drop=True)], axis=1)
yb = base[TARGET_COL].astype(int).to_numpy()
lr = LogisticRegression(max_iter=2000, C=1.0).fit(Xb, yb)

def lr_demand_elasticity(delta=DELTA):
    a, price = base["area"].to_numpy(), base[PRICE_COL].to_numpy()
    def q(mult):
        Xt = Xb.copy(); Xt["log_price"] = np.log1p(price * mult)
        return (lr.predict_proba(Xt)[:, 1] * a).sum()
    Q0 = q(1.0)
    return (q(1 + delta) - q(1 - delta)) / (Q0 * 2 * delta)

e_cat, _ = demand_elasticity(base)
e_lr = lr_demand_elasticity()
print(f"CatBoost (area-взвеш. ε, ±5%):  {e_cat:.3f}")
print(f"Logit-бейзлайн (та же метрика): {e_lr:.3f}")
print(f"коэф. log_price в логите: {lr.coef_[0][0]:.3f}  (отрицательный → дороже → ниже P)")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Logit\nбейзлайн", "CatBoost"], [e_lr, e_cat], color=[C3, C1])
ax.axhline(-1, color=C2, ls="--", lw=1, label="ε=−1")
ax.set_ylabel("area-взвеш. ε (±5%)"); ax.set_title("Эластичность: бейзлайн vs модель"); ax.legend()
plt.tight_layout(); fig.savefig(FIG_DIR / "fig03_baseline_vs_model.png", bbox_inches="tight"); plt.show()

## 4. Сохранение

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
dyn.to_csv(PROCESSED_DIR / "elasticity_dynamics.csv", index=False)
seg.drop(columns=["qm","qp"]).to_csv(PROCESSED_DIR / "elasticity_segments_detailed.csv", index=False)
print("Сохранено: elasticity_dynamics.csv, elasticity_segments_detailed.csv")
print(f"итог: ε рынка(опорный срез) CatBoost {e_cat:.3f} vs logit {e_lr:.3f}; "
      f"corr(ε, ключевая ставка) {dyn['elast'].corr(dyn['key_rate']):.2f}")